# optuna_final_0_01: trial 1647 network breakdown

Loads the network generated from Optuna trial 1647, the best trial in `data/optuna_final_0_01.db`.

1. **IZ model breakdown**: neuron counts, population fractions, Izhikevich parameters, stimulation and background currents
2. **Connection breakdown**: connection probabilities plus the per-pair connection parameters (`g`, `cond`, `delay`, `prop`, `range_*`)
3. **Cross-check**: the values stored in the generated-network pickle against the Optuna trial parameters

Run from the repo root (the folder that contains `data/`).

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

REPO = Path(".").resolve()
DATA = REPO / "data"
DB = DATA / "optuna_final_0_01.db"
STUDY_NAME = "optuna_final_0_01"
BEST_TRIAL = 1647
RUN_DIR = DATA / "optuna_final_0_01_1647"

assert DB.exists(), f"Optuna DB not found at {DB}; run from the repo root"
PARAMS_PKL = next(RUN_DIR.glob("optuna_final_0_01_1647_params_*.pkl"))
print("DB:", DB)
print("Network pickle:", PARAMS_PKL.name)

## Loading the pickle

The pickled `Params` class was defined in a script that isn't in this repo, so the pickle can't be loaded directly. This loader stands in for missing classes and keeps their attributes.

In [ ]:
class _Stub:
    def __setstate__(self, state):
        if isinstance(state, dict):
            self.__dict__.update(state)
        else:
            self.__dict__["_state"] = state


class _StubUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except (AttributeError, ImportError):
            return type(name, (_Stub,), {"_source_module": module})


def load_pickle(path):
    with open(path, "rb") as fh:
        return _StubUnpickler(fh).load()


net = load_pickle(PARAMS_PKL)
print("Attributes:", list(vars(net).keys()))
print("simTime (ms):", net.simTime)

## Optuna trial 1647

The study holds 4000 trials. The best trial should be 1647 with a value of about 0.894.

In [ ]:
study = optuna.load_study(study_name=STUDY_NAME, storage=f"sqlite:///{DB.as_posix()}")
best = study.best_trial
print("Best trial:", best.number, "| value:", best.value, "| state:", best.state)
assert best.number == BEST_TRIAL, f"Expected best trial {BEST_TRIAL}, got {best.number}"

trial = study.trials[BEST_TRIAL]
print("Tuned parameters:", len(trial.params), "(51 connection probabilities + 9 currents)")

## 1. IZ model breakdown

One row per neuron group. `i_stim` is the stimulation current and `backI` is the background current.

In [ ]:
neuron_rows = []
total_pop = sum(g["pop"] for g in net.groupInfo.values())
for idx, g in net.groupInfo.items():
    row = {"group": idx, "nType": g["nType"], "ei": g["ei"], "pop": g["pop"],
           "pop_frac": g["pop"] / total_pop,
           "stimSize": g["stimSize"], "stimTime": g["stimTime"], "i_stim": g["i"], "backI": g["backI"]}
    row.update({k: g[k] for k in ["k", "a", "b", "d", "C", "Vr", "Vt", "Vmin", "Vpeak"]})
    neuron_rows.append(row)

neurons = pd.DataFrame(neuron_rows).set_index("group")
print(f"Total neurons: {total_pop:,}")
neurons.style.format({"pop_frac": "{:.2%}", "i_stim": "{:.2f}", "backI": "{:.2f}",
                      "stimSize": "{:.1f}", "a": "{:.5f}", "b": "{:.3f}", "k": "{:.3f}",
                      "d": "{:.1f}", "C": "{:.1f}", "Vr": "{:.2f}", "Vt": "{:.2f}",
                      "Vmin": "{:.2f}", "Vpeak": "{:.2f}"})

## 2. Connection breakdown

One row per connection pair in the network (51 of the 64 possible pairs). `prob` is the connection probability from the pickle and `prop` is the base probability. `g`, `cond`, `delay`, `range_low`, `range_med` and `range_high` are the per-pair values stored with the network.

In [ ]:
conn_rows = []
for (pre, post), c in net.connInfo.items():
    conn_rows.append({
        "pre": pre, "pre_type": net.groupInfo[pre]["nType"],
        "post": post, "post_type": net.groupInfo[post]["nType"],
        "prob": c["prob"], "prop": c["prop"],
        "g": c["g"], "cond": c["cond"], "delay": c["delay"],
        "range_low": c["range_low"], "range_med": c["range_med"], "range_high": c["range_high"],
    })

connections = pd.DataFrame(conn_rows).sort_values(["pre", "post"]).reset_index(drop=True)
print(f"Connection pairs: {len(connections)}")
connections

### Connection probability matrix

Rows are presynaptic groups and columns are postsynaptic groups. Blank cells have no connection.

In [ ]:
names = [net.groupInfo[i]["nType"] for i in range(len(net.groupInfo))]
prob_matrix = pd.DataFrame(np.nan, index=names, columns=names)
prob_matrix.index.name = "pre \\ post"
for _, r in connections.iterrows():
    prob_matrix.iloc[r["pre"], r["post"]] = r["prob"]

prob_matrix.style.format("{:.4f}", na_rep="").background_gradient(cmap="viridis", axis=None)

## 3. Cross-check against the Optuna trial

The 51 connection probabilities and the 9 currents in trial 1647 should match the generated network. A mismatch means the pickle and the trial come from different runs.

In [ ]:
diffs = []
for (pre, post), c in net.connInfo.items():
    name = f"{net.groupInfo[pre]['nType']}->{net.groupInfo[post]['nType']} p"
    diffs.append(abs(trial.params[name] - c["prob"]))
max_prob_diff = max(diffs)

current_pairs = [("Pyramidal stimulation current", net.groupInfo[0]["i"])]
current_pairs += [(f"{g['nType']} background current", g["backI"]) for g in net.groupInfo.values()]
max_current_diff = max(abs(trial.params[k] - v) for k, v in current_pairs)

print(f"Max |trial prob - pickle prob|:       {max_prob_diff:.2e}")
print(f"Max |trial current - pickle current|: {max_current_diff:.2e}")
assert max_prob_diff < 1e-12 and max_current_diff < 1e-9, "Trial and network pickle disagree"
print("OK: trial 1647 matches the generated network")

## Save to CSV

Writes the neuron and connection tables next to the trial 1647 run files.

In [ ]:
OUT_DIR = RUN_DIR
neurons_csv = OUT_DIR / f"optuna_final_0_01_1647_neurons.csv"
connections_csv = OUT_DIR / f"optuna_final_0_01_1647_connections.csv"
neurons.to_csv(neurons_csv)
connections.to_csv(connections_csv, index=False)
print("Wrote", neurons_csv)
print("Wrote", connections_csv)
